# Superdense syndrome extraction for non-self-dual codes
Code A: rotated surface code. Code B: [[72,12,6]] bivariate bicycle code. Decoder: BP-OSD. Noise: uniform circuit-level depolarizing.

Run the cells from top to bottom. The heavy BB cells are meant for the server (5 GiB limit, 6 workers).

In [1]:
!pip install stim~=1.14
!pip install numpy~=1.0  # numpy 1.x, as installed on the server
!pip install scipy
!pip install -U ldpc

## Task 2: circuits
GF(2) helpers, the two codes, the superdense pairing and the circuit builder.

In [2]:
"""Memory-experiment circuits: bare-ancilla (interleaved) vs superdense syndrome extraction.
Codes: rotated surface code (A) and bivariate bicycle code (B). Noise: uniform circuit-level depolarizing."""
import random
import numpy as np
import stim

# GF(2) helpers. GF(2) is the field with the two elements 0 and 1 (arithmetic mod 2).
# The support of a stabilizer is stored as a binary vector (1 = the stabilizer acts on that qubit),
# so checks and logical operators become linear algebra mod 2.
# np.linalg.matrix_rank works over the real numbers, not mod 2, so we write our own row reduction.


def _rref(M):
    """Reduced row-echelon form of M (mod 2). Returns the reduced matrix and the pivot columns."""
    M = np.array(M, dtype=np.uint8) % 2
    pivots = []
    row = 0
    for col in range(M.shape[1]):
        # find a row (at or below `row`) with a 1 in this column
        p = None
        for i in range(row, M.shape[0]):
            if M[i, col]:
                p = i
                break
        if p is None:
            continue
        M[[row, p]] = M[[p, row]]
        # clear this column in every other row
        for i in range(M.shape[0]):
            if i != row and M[i, col]:
                M[i] ^= M[row]
        pivots.append(col)
        row += 1
        if row == M.shape[0]:
            break
    return M, pivots


def rank(M):
    """Rank of a binary matrix mod 2 (= number of independent stabilizer generators)."""
    return len(_rref(M)[1])


def nullspace(M):
    """Basis of all binary vectors v with M v = 0 (mod 2)."""
    R, pivots = _rref(M)
    basis = []
    free_cols = [j for j in range(R.shape[1]) if j not in pivots]
    for f in free_cols:
        v = np.zeros(R.shape[1], dtype=np.uint8)
        v[f] = 1
        for k, pc in enumerate(pivots):
            if R[k, f]:
                v[pc] = 1
        basis.append(v)
    return basis


def mat(checks, n):
    """List of stabilizer supports -> binary check matrix with n columns."""
    H = np.zeros((len(checks), n), dtype=np.uint8)
    for i, support in enumerate(checks):
        H[i, support] = 1
    return H


def logicals(code, basis):
    """Independent logical operators of type `basis` ('Z' for a Z-memory experiment, 'X' for X-memory).

    A Z-type logical must commute with all X checks (it is in the nullspace of Hx) and must not be
    a product of Z checks. We go through the nullspace and keep a vector only if it is independent
    of the Z checks and of the logicals already kept (the rank goes up by one)."""
    n = code['n']
    Hx, Hz = mat(code['X'], n), mat(code['Z'], n)
    if basis == 'Z':
        commute_with, stabs = Hx, Hz
    else:
        commute_with, stabs = Hz, Hx
    current, r = stabs.copy(), rank(stabs)
    found = []
    for v in nullspace(commute_with):
        test = np.vstack([current, v])
        if rank(test) > r:
            current, r = test, r + 1
            found.append(np.flatnonzero(v).tolist())
    return found

In [3]:
# Code definitions. A code is a dict with:
#   n      number of data qubits
#   X, Z   lists of check supports (data-qubit indices)
#   xs, zs the bare-ancilla schedule: for every check a list of (data qubit, time slot)


def surface_code(d):
    """Rotated surface code of distance d. Data qubit (r, c) has index r * d + c."""
    X, Z, xs, zs = [], [], [], []
    # Time slot of each plaquette corner (0 = top-left, 1 = top-right, 2 = bottom-left, 3 = bottom-right).
    # X and Z checks use different orders (Z-shape vs N-shape), the standard hook-safe choice.
    x_slot = {0: 1, 1: 2, 2: 3, 3: 4}
    z_slot = {0: 1, 2: 2, 1: 3, 3: 4}
    for i in range(-1, d):
        for j in range(-1, d):
            corners = [(i, j), (i, j + 1), (i + 1, j), (i + 1, j + 1)]
            inside = [(k, r * d + c) for k, (r, c) in enumerate(corners) if 0 <= r < d and 0 <= c < d]
            kind = 'X' if (i + j) % 2 == 0 else 'Z'
            on_top_bottom = i in (-1, d - 1)
            on_left_right = j in (-1, d - 1)
            is_bulk = len(inside) == 4
            is_boundary = len(inside) == 2 and ((on_top_bottom and kind == 'X') or (on_left_right and kind == 'Z'))
            if is_bulk or is_boundary:
                support = [q for _, q in inside]
                if kind == 'X':
                    X.append(support)
                    xs.append([(q, x_slot[k]) for k, q in inside])
                else:
                    Z.append(support)
                    zs.append([(q, z_slot[k]) for k, q in inside])
    return dict(name=f'surface_d{d}', n=d * d, X=X, Z=Z, xs=xs, zs=zs)


def bb_code(l=6, m=6, A=((3, 0), (0, 1), (0, 2)), B=((0, 3), (1, 0), (2, 0))):
    """Bivariate bicycle code, [[72,12,6]] by default (A = x^3 + y + y^2, B = y^3 + x + x^2).
    X check = [A|B], Z check = [B^T|A^T]. Qubits: left block (0..lm-1), then right block."""
    n = l * m

    def L(i, j):
        return (i % l) * m + j % m

    def R(i, j):
        return n + (i % l) * m + j % m

    X, Z = [], []
    for i in range(l):
        for j in range(m):
            X.append([L(i + a, j + b) for a, b in A] + [R(i + a, j + b) for a, b in B])
            Z.append([L(i - a, j - b) for a, b in B] + [R(i - a, j - b) for a, b in A])
    return dict(name=f'bb_{2 * n}', n=2 * n, X=X, Z=Z, l=l, m=m)


def find_bb_schedule(code, D=7, tries=400000, seed=0):
    """Random search for an interleaved bare schedule of the BB code.
    Each of the 6 terms of a check (0-2: left block, 3-5: right block) gets a time slot 1..D,
    the same for every check (translation invariant). Conditions:
      (1) an X and a Z term may share a slot only if they act on different blocks (no qubit clash);
      (2) for every overlapping X/Z check pair, X acts before Z on the left overlap qubit
          exactly when it does on the right one (otherwise the checks are measured wrong)."""
    rng = random.Random(seed)
    slots = list(range(1, D + 1))
    for _ in range(tries):
        tx = rng.sample(slots, 6)
        tz = rng.sample(slots, 6)
        clash = any(tx[a] == tz[b] and a // 3 == b // 3 for a in range(6) for b in range(6))
        if clash:
            continue
        consistent = all((tx[i] < tz[j]) == (tx[3 + j] < tz[3 + i]) for i in range(3) for j in range(3))
        if consistent:
            code['xs'] = [[(q, tx[k]) for k, q in enumerate(s)] for s in code['X']]
            code['zs'] = [[(q, tz[k]) for k, q in enumerate(s)] for s in code['Z']]
            return tx, tz
    raise RuntimeError('no schedule found')

In [4]:
# Superdense pairing: every X check is paired with one Z check, and each pair shares one Bell pair of ancillas.


def pairs_bb(code, shift=(3, 3)):
    """BB code: pair the X check at (i, j) with the Z check at (i + shift[0], j + shift[1]) (mod l, m)."""
    l, m = code['l'], code['m']
    pairs = []
    for i in range(l):
        for j in range(m):
            x_check = i * m + j
            z_check = ((i + shift[0]) % l) * m + (j + shift[1]) % m
            pairs.append((x_check, z_check))
    return pairs


def overlap_graph(code):
    """For every X check, the list of Z checks it shares at least one data qubit with."""
    X, Z = code['X'], code['Z']
    return [[z for z in range(len(Z)) if set(X[x]) & set(Z[z])] for x in range(len(X))]


def perfect_matching(adj, order):
    """Augmenting-path bipartite matching (Kuhn's algorithm). adj[x] = allowed Z checks of X check x.
    Returns a sorted list of (x, z) pairs."""
    match = {}  # z -> x

    def try_assign(x, seen):
        for z in adj[x]:
            if z in seen:
                continue
            seen.add(z)
            if z not in match or try_assign(match[z], seen):
                match[z] = x
                return True
        return False

    for x in order:
        assert try_assign(x, set()), 'no perfect matching'
    return sorted((x, z) for z, x in match.items())


def pairs_matching(code):
    """Surface code: a perfect matching between X and Z checks that overlap."""
    adj = overlap_graph(code)
    return perfect_matching(adj, range(len(adj)))


def assign(P, Q):
    """Which data qubits ancilla a handles (the rest of P∪Q goes to ancilla b).
    a is the X-check ancilla, so it gets all of P that is not in Q; b gets all of Q that is not in P.
    The overlap qubits alternate between a and b. This way only existing check-ancilla couplers are used."""
    overlap = [q for q in P if q in Q]
    A = [q for q in P if q not in overlap] + overlap[0::2]
    return set(A)


def pack(ops):
    """First-fit layering of CNOTs: each CNOT goes into the first layer where neither of its qubits is busy."""
    layers = []
    for op in ops:
        _, c, t = op
        for layer in layers:
            busy = {q for _, c2, t2 in layer for q in (c2, t2)}
            if c not in busy and t not in busy:
                layer.append(op)
                break
        else:
            layers.append([op])
    return layers

In [5]:
# Circuit builder.


class CircuitBuilder:
    """Adds one layer (TICK) of operations at a time, together with its noise.

    Noise model (uniform circuit-level depolarizing with strength p):
      reset        -> X_ERROR(p) after R, Z_ERROR(p) after RX
      CNOT         -> DEPOLARIZE2(p) on both qubits
      measurement  -> result flipped with probability p
      idle qubit   -> DEPOLARIZE1(p) on every qubit that does nothing in the layer
    `weights` multiplies p per channel, e.g. dict(idle=0) switches idle noise off."""

    def __init__(self, num_qubits, p, weights=None):
        self.circuit = stim.Circuit()
        self.num_qubits = num_qubits
        self.p = p
        self.num_measurements = 0
        self.last_meas = {}  # qubit -> index of its most recent measurement
        self.w = dict(cx=1, idle=1, reset=1, meas=1)
        if weights:
            self.w.update(weights)

    def rec(self, index):
        """stim measurement-record target for an absolute measurement index."""
        return stim.target_rec(index - self.num_measurements)

    def tick(self, ops):
        """ops: list of ('R', q), ('RX', q), ('CX', control, target), ('M', q) or ('MX', q)."""
        c, p, used = self.circuit, self.p, set()
        for kind, error in (('R', 'X_ERROR'), ('RX', 'Z_ERROR')):
            qs = [op[1] for op in ops if op[0] == kind]
            if qs:
                c.append(kind, qs)
                c.append(error, qs, p * self.w['reset'])
                used.update(qs)
        cx = [q for op in ops if op[0] == 'CX' for q in op[1:]]
        if cx:
            c.append('CX', cx)
            c.append('DEPOLARIZE2', cx, p * self.w['cx'])
            used.update(cx)
        for kind in ('M', 'MX'):
            qs = [op[1] for op in ops if op[0] == kind]
            if qs:
                c.append(kind, qs, p * self.w['meas'])
                used.update(qs)
                for q in qs:
                    self.last_meas[q] = self.num_measurements
                    self.num_measurements += 1
        idle = [q for q in range(self.num_qubits) if q not in used]
        if idle:
            c.append('DEPOLARIZE1', idle, p * self.w['idle'])
        c.append('TICK')


def superdense_layers(code, pairs, n):
    """Default superdense schedule (used when build() gets no `schedule`):
    all Z-accumulation CNOTs, then all X-accumulation CNOTs, each packed first-fit.
    Returns (list of CNOT layers, feedback list)."""
    X, Z = code['X'], code['Z']
    z_ops, x_ops, feedback = [], [], []
    for k, (x, z) in enumerate(pairs):
        a, b = n + 2 * k, n + 2 * k + 1
        P, Q = X[x], Z[z]
        A = assign(P, Q)
        for d in Q:   # Z accumulation: data qubit is the control
            z_ops.append(('CX', d, a if d in A else b))
        for d in P:   # X accumulation: ancilla is the control
            x_ops.append(('CX', a if d in A else b, d))
        # kickback: X on the data qubits that a controls, applied if the Z-check outcome is -1
        feedback.append([d for d in P if d in A])
    return pack(z_ops) + pack(x_ops), feedback


def build(code, mode, rounds, basis='Z', p=1e-3, pairs=None, noise=None, schedule=None):
    """Memory experiment with `rounds` rounds of syndrome extraction.
    mode: 'bare' (one ancilla per check, interleaved schedule code['xs'], code['zs'])
          or 'superdense' (one Bell pair per (X check, Z check) pair in `pairs`).
    schedule: optional (cnot_layers, feedback) for superdense, e.g. from make_schedule().
    noise: optional per-channel weights, see CircuitBuilder."""
    n, X, Z = code['n'], code['X'], code['Z']
    nx, nz = len(X), len(Z)
    data = list(range(n))

    if mode == 'bare':
        num_qubits = n + nx + nz
        x_anc = [n + i for i in range(nx)]
        z_anc = [n + nx + i for i in range(nz)]
    else:
        # pair k uses ancillas a = n + 2k (X check) and b = n + 2k + 1 (Z check)
        num_qubits = n + 2 * len(pairs)
        x_anc, z_anc, bell_pairs = [None] * nx, [None] * nz, []
        for k, (x, z) in enumerate(pairs):
            x_anc[x], z_anc[z] = n + 2 * k, n + 2 * k + 1
            bell_pairs.append((n + 2 * k, n + 2 * k + 1))
        if schedule is None:
            cnot_layers, feedback = superdense_layers(code, pairs, n)
        else:
            cnot_layers, feedback = schedule

    B = CircuitBuilder(num_qubits, p, noise)
    B.tick([('R' if basis == 'Z' else 'RX', q) for q in data])

    def one_round():
        if mode == 'bare':
            B.tick([('RX', a) for a in x_anc] + [('R', a) for a in z_anc])
            depth = max(t for s in code['xs'] + code['zs'] for _, t in s)
            for t in range(1, depth + 1):
                layer = []
                for k, s in enumerate(code['xs']):
                    layer += [('CX', x_anc[k], q) for q, slot in s if slot == t]
                for k, s in enumerate(code['zs']):
                    layer += [('CX', q, z_anc[k]) for q, slot in s if slot == t]
                B.tick(layer)
            B.tick([('MX', a) for a in x_anc] + [('M', a) for a in z_anc])
        else:
            B.tick([('RX', a) for a, _ in bell_pairs] + [('R', b) for _, b in bell_pairs])
            B.tick([('CX', a, b) for a, b in bell_pairs])           # make the Bell pair
            for layer in cnot_layers:
                B.tick(layer)
            B.tick([('CX', a, b) for a, b in bell_pairs])           # undo it (Bell measurement)
            B.tick([('MX', a) for a, _ in bell_pairs] + [('M', b) for _, b in bell_pairs])
            # Pauli feedback, classically controlled by the Z-check result (removed again below)
            for (a, b), fb in zip(bell_pairs, feedback):
                for d in fb:
                    B.circuit.append('CX', [B.rec(B.last_meas[b]), d])
        results = {}
        for k in range(nx):
            results[('X', k)] = B.last_meas[x_anc[k]]
        for k in range(nz):
            results[('Z', k)] = B.last_meas[z_anc[k]]
        return results

    prev = None
    for r in range(rounds):
        cur = one_round()
        for (kind, k), meas in cur.items():
            if prev is None and kind != basis:
                continue   # first round: only checks of the memory basis are deterministic
            targets = [B.rec(meas)]
            if prev is not None:
                targets.append(B.rec(prev[(kind, k)]))
            B.circuit.append('DETECTOR', targets, [k, 0, r])
        prev = cur

    # final data measurement, last detectors and the logical observables
    B.tick([('M' if basis == 'Z' else 'MX', q) for q in data])
    final_checks = Z if basis == 'Z' else X
    for k, s in enumerate(final_checks):
        targets = [B.rec(B.last_meas[q]) for q in s] + [B.rec(prev[(basis, k)])]
        B.circuit.append('DETECTOR', targets, [k, 0, rounds])
    for i, lg in enumerate(logicals(code, basis)):
        B.circuit.append('OBSERVABLE_INCLUDE', [B.rec(B.last_meas[q]) for q in lg], i)
    # stim moves the classically controlled feedback into the detector/observable definitions
    return B.circuit.with_inlined_feedback()

## Task 3: BP-OSD benchmarks

In [6]:
"""BP-OSD benchmarking of the circuits built above."""
import time
import resource
import numpy as np
import scipy.sparse as sp
from concurrent.futures import ProcessPoolExecutor, as_completed
from ldpc import BpOsdDecoder


def dem_matrices(circ):
    """Detector error model -> (H, L, probs).
    H: detectors x error mechanisms, L: observables x error mechanisms, probs: probability of each mechanism.
    Mechanisms with the same detectors and observables are merged (independent XOR of their probabilities)."""
    dem = circ.detector_error_model(decompose_errors=False)
    merged = {}
    for inst in dem.flattened():
        if inst.type != 'error':
            continue
        p = inst.args_copy()[0]
        targets = inst.targets_copy()
        dets = frozenset(t.val for t in targets if t.is_relative_detector_id())
        obs = frozenset(t.val for t in targets if t.is_logical_observable_id())
        q = merged.get((dets, obs), 0.0)
        merged[(dets, obs)] = q * (1 - p) + p * (1 - q)
    keys = list(merged)
    H = sp.lil_matrix((dem.num_detectors, len(keys)), dtype=np.uint8)
    L = sp.lil_matrix((dem.num_observables, len(keys)), dtype=np.uint8)
    for j, (dets, obs) in enumerate(keys):
        for i in dets:
            H[i, j] = 1
        for i in obs:
            L[i, j] = 1
    return H.tocsc(), L.tocsc(), np.array([merged[k] for k in keys])


def make_decoder(H, probs, max_iter=30, osd_order=7):
    return BpOsdDecoder(H, error_channel=list(probs), max_iter=max_iter, bp_method='minimum_sum',
                        ms_scaling_factor=0.625, osd_method='osd_cs', osd_order=osd_order)


def bposd_ler(circ, shots, max_iter=30, osd_order=7, seed=None):
    """Single-process benchmark. Returns (failures, shots, syndrome mismatches).
    A shot fails if any logical observable is predicted wrong."""
    H, L, probs = dem_matrices(circ)
    dec = make_decoder(H, probs, max_iter, osd_order)
    det, obs = circ.compile_detector_sampler(seed=seed).sample(shots, separate_observables=True)
    fails = bad = 0
    for s, o in zip(det.astype(np.uint8), obs):
        e = dec.decode(s)
        bad += int(((H @ e) % 2 != s).any())      # the correction must reproduce the syndrome
        fails += int(((L @ e) % 2 != o).any())
    return fails, shots, bad


def report(fails, shots):
    """LER and its standard error."""
    p = fails / shots
    return p, np.sqrt(max(p * (1 - p), 1e-12) / shots)


# ---- parallel runner ----
# Each worker builds the decoder once and keeps it, then decodes chunks of shots.
# Workers are forked from the notebook (Linux server), so they see the functions defined above.
# Memory: about 0.5 GB per worker for the BB circuits; keep workers * 0.5 GB below your limit (5 GiB).
_decoder_cache = {}


def _run_chunk(args):
    circuit_args, shots, seed, max_iter, osd_order = args
    key = (repr(circuit_args), max_iter, osd_order)
    if key not in _decoder_cache:
        _decoder_cache.clear()
        code, mode, rounds, basis, p, pairs, noise, schedule = circuit_args
        circ = build(code, mode, rounds, basis, p, pairs, noise=noise, schedule=schedule)
        H, L, probs = dem_matrices(circ)
        _decoder_cache[key] = (circ, L, make_decoder(H, probs, max_iter, osd_order))
    circ, L, dec = _decoder_cache[key]
    det, obs = circ.compile_detector_sampler(seed=seed).sample(shots, separate_observables=True)
    fails = 0
    for s, o in zip(det.astype(np.uint8), obs):
        e = dec.decode(s)
        fails += int(((L @ e) % 2 != o).any())
    peak_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss // 1024
    return fails, shots, peak_mb


def run_point(code, mode, pairs, rounds, basis, p, shots, workers=6, chunk=25,
              noise=None, schedule=None, max_iter=30, osd_order=7):
    """Parallel benchmark of one circuit. Returns (failures, shots, standard error).
    Shows progress and peak worker memory; a killed worker raises BrokenProcessPool instead of hanging."""
    circuit_args = (code, mode, rounds, basis, p, pairs, noise, schedule)
    start = time.time()
    fails = done = peak = 0
    with ProcessPoolExecutor(workers) as ex:
        jobs = [ex.submit(_run_chunk, (circuit_args, chunk, 10_000 + i, max_iter, osd_order))
                for i in range(shots // chunk)]
        for job in as_completed(jobs):
            f, n, mem = job.result()
            fails += f
            done += n
            peak = max(peak, mem)
            print(f'\r{done}/{shots} shots, {fails} failures, peak worker RAM {peak} MB, '
                  f'{time.time() - start:.0f}s', end='', flush=True)
    print()
    return fails, done, report(fails, done)[1]

### Surface code

In [7]:
shots = {3: 20000, 5: 8000}          # d=7 was too slow
print('d  p     mode        LER ± se')
for d in (3, 5):
    S = surface_code(d)
    P = pairs_matching(S)
    for p in (1e-3, 2e-3, 4e-3):
        for mode in ('bare', 'superdense'):
            f, n, bad = bposd_ler(build(S, mode, d, 'Z', p, P), shots[d], seed=1)
            ler, se = report(f, n)
            print(d, f'{p:.0e}', mode, f'{ler:.4f} ± {se:.4f}', 'mismatches:', bad)

d  p     mode        LER ± se
3 1e-03 bare 0.0014 ± 0.0003 mismatches: 0
3 1e-03 superdense 0.0060 ± 0.0005 mismatches: 0
3 2e-03 bare 0.0066 ± 0.0006 mismatches: 0
3 2e-03 superdense 0.0210 ± 0.0010 mismatches: 0
3 4e-03 bare 0.0204 ± 0.0010 mismatches: 0
3 4e-03 superdense 0.0660 ± 0.0018 mismatches: 0
5 1e-03 bare 0.0004 ± 0.0002 mismatches: 0
5 1e-03 superdense 0.0021 ± 0.0005 mismatches: 0
5 2e-03 bare 0.0010 ± 0.0004 mismatches: 0
5 2e-03 superdense 0.0144 ± 0.0013 mismatches: 0
5 4e-03 bare 0.0143 ± 0.0013 mismatches: 0
5 4e-03 superdense 0.0917 ± 0.0032 mismatches: 0


### BB code
Sweep, decoder-strength check and circuit-distance check.

In [9]:
B = bb_code()
find_bb_schedule(B, D=7)
P = pairs_bb(B)

In [8]:
B = bb_code()
find_bb_schedule(B, D=7)
P = pairs_bb(B)
for p in (2e-3, 3e-3, 4e-3):
    for mode in ('bare', 'superdense'):
        f, n, se = run_point(B, mode, P, 6, 'Z', p, shots=2000, workers=6, osd_order=5, max_iter=30)
        print(p, mode, f, n, round(f / n, 4), round(se, 4))

2000/2000 shots, 88 failures, peak worker RAM 350 MB, 726s
0.002 bare 88 2000 0.044 0.0046
2000/2000 shots, 599 failures, peak worker RAM 495 MB, 1392s
0.002 superdense 599 2000 0.2995 0.0102
2000/2000 shots, 298 failures, peak worker RAM 352 MB, 790s
0.003 bare 298 2000 0.149 0.008
2000/2000 shots, 1325 failures, peak worker RAM 489 MB, 1570s
0.003 superdense 1325 2000 0.6625 0.0106
2000/2000 shots, 750 failures, peak worker RAM 350 MB, 852s
0.004 bare 750 2000 0.375 0.0108
2000/2000 shots, 1824 failures, peak worker RAM 498 MB, 1684s
0.004 superdense 1824 2000 0.912 0.0063


In [10]:
# Decoder-strength check at p=2e-3: if a stronger decoder lowered the LER a lot, the settings above would be too weak.
settings = {'bare': ((30, 5), (100, 5), (100, 10), (300, 10)), 'superdense': ((30, 5), (100, 10))}
for mode in ('bare', 'superdense'):
    for it, osd in settings[mode]:
        f, n, se = run_point(B, mode, P, 6, 'Z', 2e-3, shots=480, workers=6, max_iter=it, osd_order=osd)
        print(mode, it, osd, f, n, round(f / n, 4))

475/480 shots, 18 failures, peak worker RAM 354 MB, 192s
bare 30 5 18 475 0.0379
475/480 shots, 19 failures, peak worker RAM 351 MB, 198s
bare 100 5 19 475 0.04
475/480 shots, 19 failures, peak worker RAM 354 MB, 198s
bare 100 10 19 475 0.04
475/480 shots, 15 failures, peak worker RAM 350 MB, 223s
bare 300 10 15 475 0.0316
475/480 shots, 148 failures, peak worker RAM 502 MB, 369s
superdense 30 5 148 475 0.3116
475/480 shots, 150 failures, peak worker RAM 500 MB, 375s
superdense 100 10 150 475 0.3158


In [11]:
# Circuit-level distance of the BB circuits (bounded search: an upper bound, not a proof)
for mode in ('bare', 'superdense'):
    for basis in 'ZX':
        c = build(B, mode, 3, basis, 1e-3, P)
        e = c.search_for_undetectable_logical_errors(
            dont_explore_detection_event_sets_with_size_above=4, dont_explore_edges_with_degree_above=4,
            dont_explore_edges_increasing_symptom_degree=False, canonicalize_circuit_errors=True)
        print(mode, basis, 'smallest logical fault found:', len(e))

bare Z smallest logical fault found: 6
bare X smallest logical fault found: 6
superdense Z smallest logical fault found: 6
superdense X smallest logical fault found: 6


## Task 4: alternative superdense circuits
Search, validation and checklist functions.

In [12]:
import time
from collections import Counter

# A superdense schedule is a pair (cnot_layers, feedback):
#   cnot_layers  list of layers, each a list of ('CX', control, target) on data qubits and ancillas
#   feedback     feedback[k] = data qubits that get X when the Z-check result of pair k is -1
#                (= the data qubits that ancilla a of pair k controls during X accumulation)


def random_pairs(code, rng):
    """Random perfect matching between overlapping X and Z checks."""
    adj = overlap_graph(code)
    for a in adj:
        rng.shuffle(a)
    order = list(range(len(adj)))
    rng.shuffle(order)
    return perfect_matching(adj, order)


def make_schedule(code, pairs, rng, mode='twophase'):
    """Random superdense schedule for a given pairing.

    Ancilla a gets the X-only qubits of its pair and b the Z-only ones (only existing couplers);
    each overlap qubit goes to a or b at random. The order of the CNOTs of each ancilla is random
    ('_ranked' modes: ordered by a random global rank per data qubit, which packs better).
    The CNOTs are then placed greedily into the earliest free layer.
      twophase: all Z accumulations of all pairs come before any X accumulation (covered by the task 1 proof)
      pairwise: an ancilla starts X accumulation as soon as its own pair finished Z accumulation
                (shorter, but only validated by simulation)"""
    n, X, Z = code['n'], code['X'], code['Z']
    ranked = mode.endswith('_ranked')
    mode = mode.replace('_ranked', '')
    z_rank = {q: rng.random() for q in range(n)}
    x_rank = {q: rng.random() for q in range(n)}

    sequences = {}   # ancilla -> (list of Z-accumulation ops, list of X-accumulation ops)
    pair_of = {}     # ancilla -> pair index
    feedback = []
    for k, (x, z) in enumerate(pairs):
        a, b = n + 2 * k, n + 2 * k + 1
        P, Q = X[x], Z[z]
        overlap = [q for q in P if q in Q]
        A = {q for q in P if q not in overlap} | {q for q in overlap if rng.random() < 0.5}
        z_targets = {a: [], b: []}
        x_targets = {a: [], b: []}
        for d in Q:
            z_targets[a if d in A else b].append(d)
        for d in P:
            x_targets[a if d in A else b].append(d)
        for anc in (a, b):
            if ranked:
                z_targets[anc].sort(key=z_rank.get)
                x_targets[anc].sort(key=x_rank.get)
            else:
                rng.shuffle(z_targets[anc])
                rng.shuffle(x_targets[anc])
            pair_of[anc] = k
            sequences[anc] = ([('Z', anc, d) for d in z_targets[anc]],
                              [('X', anc, d) for d in x_targets[anc]])
        feedback.append([d for d in P if d in A])

    layers, busy = [], []

    def place(op, earliest):
        """Put one CNOT into the first layer >= earliest where both its qubits are free."""
        kind, anc, d = op
        control, target = (d, anc) if kind == 'Z' else (anc, d)
        L = earliest
        while True:
            while L >= len(layers):
                layers.append([])
                busy.append(set())
            if control not in busy[L] and target not in busy[L]:
                break
            L += 1
        layers[L].append(('CX', control, target))
        busy[L].update((control, target))
        return L

    def schedule_phase(phase, earliest_for):
        """phase 0 = Z accumulation, 1 = X accumulation. Ops are placed position by position
        (all first CNOTs, then all second CNOTs, ...), ties broken at random."""
        last_layer, pair_end = {}, {}
        ops = []
        for anc, seq in sequences.items():
            for pos, op in enumerate(seq[phase]):
                ops.append((pos, rng.random(), op))
        ops.sort(key=lambda t: t[:2])
        for pos, _, op in ops:
            anc = op[1]
            L = place(op, max(earliest_for(anc), last_layer.get(anc, -1) + 1))
            last_layer[anc] = L
            pair_end[pair_of[anc]] = max(pair_end.get(pair_of[anc], -1), L)
        return pair_end

    z_end = schedule_phase(0, lambda anc: 0)
    all_z_done = max(z_end.values()) + 1
    if mode == 'twophase':
        schedule_phase(1, lambda anc: all_z_done)
    else:
        schedule_phase(1, lambda anc: z_end[pair_of[anc]] + 1)
    return layers, feedback


def candidate(code, spec):
    """Rebuild a surface-code candidate from its spec (pairs_seed, schedule_seed, mode) -> (pairs, schedule)."""
    pairs_seed, schedule_seed, mode = spec
    pairs = random_pairs(code, random.Random(pairs_seed))
    return pairs, make_schedule(code, pairs, random.Random(schedule_seed), mode)


# ---- validation ----

def _inject(circ, r, pauli, qubits):
    """Copy of `circ` with a deterministic Pauli error on `qubits` just before the r-th RX instruction
    (r = 1 is the start of round 2)."""
    out = stim.Circuit()
    count = -1
    for inst in circ:
        if inst.name == 'RX':
            count += 1
            if count == r:
                out.append(pauli + '_ERROR', qubits, 1)
        out.append(inst)
    return out


def _signature(circ):
    """Detection events and observable flips of one (noiseless) shot."""
    d, o = circ.compile_detector_sampler(seed=0).sample(1, separate_observables=True)
    return d[0].tolist(), o[0].tolist()


def is_valid(code, pairs, sched, bare0):
    """Noiseless determinism in both bases, and the same detection events as the bare circuit `bare0`
    (noiseless, Z memory, 3 rounds) for every single X or Z error on every data qubit."""
    try:
        for basis in 'ZX':
            circ = build(code, 'superdense', 3, basis, 0, pairs, schedule=sched)
            if circ.compile_detector_sampler().sample(20).any():
                return False
        sd0 = build(code, 'superdense', 3, 'Z', 0, pairs, schedule=sched)
    except ValueError:   # stim found a non-deterministic detector or observable
        return False
    for q in range(code['n']):
        for pauli in 'XZ':
            if _signature(_inject(bare0, 1, pauli, [q])) != _signature(_inject(sd0, 1, pauli, [q])):
                return False
    return True


def eff_distance(code, pairs, sched, rounds=3):
    """Smallest undetectable logical error stim finds (Z and X memory). This is a bounded heuristic search:
    it gives an upper bound on the circuit distance, not a proof."""
    best = 99
    for basis in 'ZX':
        circ = build(code, 'superdense', rounds, basis, 1e-3, pairs, schedule=sched)
        err = circ.search_for_undetectable_logical_errors(
            dont_explore_detection_event_sets_with_size_above=4, dont_explore_edges_with_degree_above=4,
            dont_explore_edges_increasing_symptom_degree=False, canonicalize_circuit_errors=True)
        best = min(best, len(err))
    return best


def search(code, target, mode, seconds=60, seed=0):
    """Surface code: try random (pairing, schedule) specs for `seconds`; keep those that are valid and
    reach circuit distance >= target. Returns ([(cnot layers, spec), ...] sorted, stats)."""
    bare0 = build(code, 'bare', 3, 'Z', 0)
    rng = random.Random(seed)
    stats = dict(tried=0, valid=0, ok=0)
    good = []
    start = time.time()
    while time.time() - start < seconds:
        spec = (rng.randrange(10**6), rng.randrange(10**6), mode)
        pairs, sched = candidate(code, spec)
        stats['tried'] += 1
        if not is_valid(code, pairs, sched, bare0):
            continue
        stats['valid'] += 1
        if eff_distance(code, pairs, sched) >= target:
            stats['ok'] += 1
            good.append((len(sched[0]), spec))
    return sorted(good), stats


def checklist(code, pairs, sched, target_distance):
    """Explicit checks that a superdense circuit is a valid replacement of the bare one."""
    n, X, Z = code['n'], code['X'], code['Z']
    layers, feedback = sched
    ok = {}
    cnots = [(c, t) for layer in layers for _, c, t in layer]

    qubit_lists = [[q for _, c, t in layer for q in (c, t)] for layer in layers]
    ok['1 no qubit used twice in a layer'] = all(len(set(qs)) == len(qs) for qs in qubit_lists)

    pair_of, P_of, Q_of = {}, {}, {}
    for k, (x, z) in enumerate(pairs):
        pair_of[n + 2 * k] = pair_of[n + 2 * k + 1] = k
        P_of[k], Q_of[k] = set(X[x]), set(Z[z])

    def coupler_ok(c, t):
        d, anc = (c, t) if c < n else (t, c)
        k = pair_of[anc]
        allowed = P_of[k] if anc == n + 2 * k else Q_of[k]
        return d in allowed
    ok['2 only existing data-ancilla couplers (a: own X support, b: own Z support)'] = \
        all(coupler_ok(c, t) for c, t in cnots)

    count = Counter()
    for c, t in cnots:
        if c < n:
            count[(pair_of[t], c, 'Z')] += 1     # data controls: Z accumulation
        else:
            count[(pair_of[c], t, 'X')] += 1     # ancilla controls: X accumulation
    good = True
    for k in range(len(pairs)):
        for d in P_of[k] | Q_of[k]:
            good &= count[(k, d, 'X')] == (d in P_of[k]) and count[(k, d, 'Z')] == (d in Q_of[k])
    ok['3 every data qubit: one X-CNOT iff in P, one Z-CNOT iff in Q (overlap qubits get both)'] = good

    ok['4 every X-Z pair overlaps evenly'] = all(len(P_of[k] & Q_of[k]) % 2 == 0 for k in P_of)

    ok['5 feedback set = data qubits that ancilla a controls'] = all(
        set(feedback[k]) == {t for c, t in cnots if c == n + 2 * k and t < n} for k in range(len(pairs)))

    ok['6 noiseless detectors deterministic (Z and X memory)'] = all(
        not build(code, 'superdense', 4, b, 0, pairs, schedule=sched).compile_detector_sampler().sample(50).any()
        for b in 'ZX')

    ok['7 same detection events as bare circuit for all single data errors'] = \
        is_valid(code, pairs, sched, build(code, 'bare', 3, 'Z', 0))

    sd0 = build(code, 'superdense', 3, 'Z', 0, pairs, schedule=sched)
    results = [_signature(_inject(sd0, 1, 'X', lg)) for lg in logicals(code, 'X')]
    ok['8 logical operators fire no detector and flip an observable'] = \
        all(not any(d) and any(o) for d, o in results)

    ok[f'9 circuit distance >= {target_distance} (5 rounds, Z and X memory)'] = \
        eff_distance(code, pairs, sched, 5) >= target_distance
    return ok

### Surface code (d=5)

In [13]:
# Surface code d=5: the two finalists of the search (rebuilt from their seeds), checklist, LER comparison.
# The search that found them: good, stats = search(S, 5, 'twophase_ranked', seconds=60, seed=...)
S = surface_code(5)
specs = {'C1': (185304, 340937, 'twophase_ranked'), 'C2': (256724, 864810, 'pairwise_ranked')}
cands = {name: candidate(S, spec) for name, spec in specs.items()}
for name, (pr, sc) in cands.items():
    print(name, 'data-CNOT layers:', len(sc[0]))
    for check, passed in checklist(S, pr, sc, target_distance=5).items():
        print('  ', 'PASS' if passed else 'FAIL', check)

C1 data-CNOT layers: 8
   PASS 1 no qubit used twice in a layer
   PASS 2 only existing data-ancilla couplers (a: own X support, b: own Z support)
   PASS 3 every data qubit: one X-CNOT iff in P, one Z-CNOT iff in Q (overlap qubits get both)
   PASS 4 every X-Z pair overlaps evenly
   PASS 5 feedback set = data qubits that ancilla a controls
   PASS 6 noiseless detectors deterministic (Z and X memory)
   PASS 7 same detection events as bare circuit for all single data errors
   PASS 8 logical operators fire no detector and flip an observable
   PASS 9 circuit distance >= 5 (5 rounds, Z and X memory)
C2 data-CNOT layers: 7
   PASS 1 no qubit used twice in a layer
   PASS 2 only existing data-ancilla couplers (a: own X support, b: own Z support)
   PASS 3 every data qubit: one X-CNOT iff in P, one Z-CNOT iff in Q (overlap qubits get both)
   PASS 4 every X-Z pair overlaps evenly
   PASS 5 feedback set = data qubits that ancilla a controls
   PASS 6 noiseless detectors deterministic (Z an

In [14]:
P0 = pairs_matching(S)


def surface_circuits(p, noise=None):
    circuits = {'bare': build(S, 'bare', 5, 'Z', p, noise=noise),
                'original SD': build(S, 'superdense', 5, 'Z', p, P0, noise=noise)}
    for name, (pr, sc) in cands.items():
        circuits[name] = build(S, 'superdense', 5, 'Z', p, pr, noise=noise, schedule=sc)
    return circuits


for label, p, noise in (('p=2e-3, full noise', 2e-3, None), ('p=4e-3, no idle noise', 4e-3, dict(idle=0))):
    row = []
    for name, c in surface_circuits(p, noise).items():
        f, n, bad = bposd_ler(c, 3500, seed=7)
        row.append(f'{name} {f}/{n}')
    print(label, '|', ' | '.join(row))

p=2e-3, full noise | bare 5/3500 | original SD 42/3500 | C1 71/3500 | C2 59/3500
p=4e-3, no idle noise | bare 8/3500 | original SD 30/3500 | C1 21/3500 | C2 28/3500


### BB code
Shift pairings X(i,j) ↔ Z(i+s0, j+s1) and translation-invariant CNOT schedules.

In [15]:
def bb_superdense_schedule(code, pairs, rng, D=6, tries=200000):
    """Translation-invariant superdense schedule for a BB shift pairing (same idea as find_bb_schedule).

    Every pair is a translate of pair 0, so every CNOT of pair 0 (6 Z-accumulation terms, 6 X-accumulation
    terms) gets one time slot in 0..D-1, and all pairs use the same slots. Ancilla a gets the X-only qubits
    and one overlap qubit, b gets the Z-only qubits and the other overlap qubit (chosen at random).
    Conditions on a random slot choice:
      (1) each ancilla does one CNOT per slot;
      (2) two CNOTs in the same slot act on different blocks (left/right), otherwise some data qubit clashes;
      (3) each ancilla does all its Z accumulation before its X accumulation (task 1 circuit, per ancilla);
      (4) X before Z at the left overlap qubit of any X/Z check pair iff at the right one (as for bare BB).
    Returns (cnot_layers, feedback) or None."""
    n, X, Z = code['n'], code['X'], code['Z']
    x0, z0 = pairs[0]
    P, Q = X[x0], Z[z0]
    overlap = [q for q in P if q in Q]
    a_overlap = rng.choice(overlap)
    # which ancilla does each term ('a' or 'b'); terms 0-2 are on the left block, 3-5 on the right block
    x_anc = ['b' if (q in overlap and q != a_overlap) else 'a' for q in P]
    z_anc = ['a' if q == a_overlap else 'b' for q in Q]

    for _ in range(tries):
        tx, tz = [None] * 6, [None] * 6
        for anc in 'ab':                                   # condition (1)
            ops = [('X', i) for i in range(6) if x_anc[i] == anc] + [('Z', j) for j in range(6) if z_anc[j] == anc]
            for (kind, term), slot in zip(ops, rng.sample(range(D), len(ops))):
                if kind == 'X':
                    tx[term] = slot
                else:
                    tz[term] = slot
        slot_block = [(tx[i], i // 3) for i in range(6)] + [(tz[j], j // 3) for j in range(6)]
        if len(set(slot_block)) < 12:                      # condition (2)
            continue
        order_ok = True
        for anc in 'ab':                                   # condition (3)
            last_z = max(tz[j] for j in range(6) if z_anc[j] == anc)
            first_x = min(tx[i] for i in range(6) if x_anc[i] == anc)
            if last_z > first_x:
                order_ok = False
        if not order_ok:
            continue
        if not all((tx[i] < tz[j]) == (tx[3 + j] < tz[3 + i]) for i in range(3) for j in range(3)):   # (4)
            continue

        used = sorted(set(tx + tz))
        layer_of = {slot: k for k, slot in enumerate(used)}
        layers = [[] for _ in used]
        feedback = []
        for k, (x, z) in enumerate(pairs):
            a, b = n + 2 * k, n + 2 * k + 1
            for j, q in enumerate(Z[z]):
                layers[layer_of[tz[j]]].append(('CX', q, a if z_anc[j] == 'a' else b))
            for i, q in enumerate(X[x]):
                layers[layer_of[tx[i]]].append(('CX', a if x_anc[i] == 'a' else b, q))
            feedback.append([q for i, q in enumerate(X[x]) if x_anc[i] == 'a'])
        return layers, feedback
    return None

In [16]:
# Search: every shift pairing with an even overlap, a few random schedules each.
# Keep a schedule only if it is valid and the distance search finds nothing below 6. (~20 s per schedule)
B = bb_code()
find_bb_schedule(B, D=7)
bare0_bb = build(B, 'bare', 3, 'Z', 0)
shifts = [(di, dj) for di in range(6) for dj in range(6) if len(set(B['X'][0]) & set(B['Z'][di * 6 + dj])) > 0]
print('shifts with overlapping checks:', shifts)
bb_good = []
for shift in shifts:
    pr = pairs_bb(B, shift)
    for seed in range(3):
        sc = bb_superdense_schedule(B, pr, random.Random(seed), D=6)
        if sc is None:
            print(shift, seed, 'no schedule found')
            continue
        valid = is_valid(B, pr, sc, bare0_bb)
        dist = eff_distance(B, pr, sc) if valid else None
        print(shift, seed, 'layers', len(sc[0]), 'valid', valid, 'distance', dist)
        if valid and dist >= 6:
            bb_good.append((shift, seed))
print('kept:', bb_good)

shifts with overlapping checks: [(0, 4), (0, 5), (1, 1), (1, 2), (2, 1), (2, 2), (3, 3), (4, 0), (5, 0)]
(0, 4) 0 layers 6 valid True distance 6
(0, 4) 1 layers 6 valid True distance 6
(0, 4) 2 layers 6 valid True distance 6
(0, 5) 0 layers 6 valid True distance 5
(0, 5) 1 layers 6 valid True distance 6
(0, 5) 2 layers 6 valid True distance 5
(1, 1) 0 layers 6 valid True distance 3
(1, 1) 1 layers 6 valid True distance 3
(1, 1) 2 layers 6 valid True distance 6
(1, 2) 0 layers 6 valid True distance 5
(1, 2) 1 layers 6 valid True distance 6
(1, 2) 2 layers 6 valid True distance 6
(2, 1) 0 layers 6 valid True distance 6
(2, 1) 1 layers 6 valid True distance 5
(2, 1) 2 layers 6 valid True distance 5
(2, 2) 0 layers 6 valid True distance 6
(2, 2) 1 layers 6 valid True distance 6
(2, 2) 2 layers 6 valid True distance 6
(3, 3) 0 layers 6 valid True distance 3
(3, 3) 1 layers 6 valid True distance 6
(3, 3) 2 layers 6 valid True distance 5
(4, 0) 0 layers 6 valid True distance 6
(4, 0) 1 layers

In [17]:
# Finalists from the search above, rebuilt from (shift, seed), and the full checklist (distance target 6).
bb_specs = {'B1': ((0, 4), 0), 'B2': ((4, 0), 0)}
bb_cands = {}
for name, (shift, seed) in bb_specs.items():
    pr = pairs_bb(B, shift)
    bb_cands[name] = (pr, bb_superdense_schedule(B, pr, random.Random(seed), D=6))
for name, (pr, sc) in bb_cands.items():
    print(name, 'data-CNOT layers:', len(sc[0]))
    for check, passed in checklist(B, pr, sc, target_distance=6).items():
        print('  ', 'PASS' if passed else 'FAIL', check)

B1 data-CNOT layers: 6
   PASS 1 no qubit used twice in a layer
   PASS 2 only existing data-ancilla couplers (a: own X support, b: own Z support)
   PASS 3 every data qubit: one X-CNOT iff in P, one Z-CNOT iff in Q (overlap qubits get both)
   PASS 4 every X-Z pair overlaps evenly
   PASS 5 feedback set = data qubits that ancilla a controls
   PASS 6 noiseless detectors deterministic (Z and X memory)
   PASS 7 same detection events as bare circuit for all single data errors
   PASS 8 logical operators fire no detector and flip an observable
   PASS 9 circuit distance >= 6 (5 rounds, Z and X memory)
B2 data-CNOT layers: 6
   PASS 1 no qubit used twice in a layer
   PASS 2 only existing data-ancilla couplers (a: own X support, b: own Z support)
   PASS 3 every data qubit: one X-CNOT iff in P, one Z-CNOT iff in Q (overlap qubits get both)
   PASS 4 every X-Z pair overlaps evenly
   PASS 5 feedback set = data qubits that ancilla a controls
   PASS 6 noiseless detectors deterministic (Z an

In [18]:
# LER comparison at p=2e-3 (Z memory, 6 rounds), with full noise and without idle noise. Run on the server.
# About 4 s per shot per worker for the superdense circuits: 1000 shots with 6 workers is roughly 10-15 min per circuit.
P0_bb = pairs_bb(B)
bb_circuits = {'bare': dict(mode='bare', pairs=None, schedule=None),
               'original SD': dict(mode='superdense', pairs=P0_bb, schedule=None)}
for name, (pr, sc) in bb_cands.items():
    bb_circuits[name] = dict(mode='superdense', pairs=pr, schedule=sc)
for label, noise in (('full noise', None), ('no idle noise', dict(idle=0))):
    for name, c in bb_circuits.items():
        f, n, se = run_point(B, c['mode'], c['pairs'], 6, 'Z', 2e-3, shots=1000, workers=6,
                             noise=noise, schedule=c['schedule'], osd_order=5, max_iter=30)
        print(label, '|', name, f, n, round(f / n, 4), '±', round(se, 4))

1000/1000 shots, 34 failures, peak worker RAM 661 MB, 356s
full noise | bare 34 1000 0.034 ± 0.0057
1000/1000 shots, 293 failures, peak worker RAM 665 MB, 694s
full noise | original SD 293 1000 0.293 ± 0.0144
1000/1000 shots, 70 failures, peak worker RAM 668 MB, 633s
full noise | B1 70 1000 0.07 ± 0.0081
1000/1000 shots, 65 failures, peak worker RAM 668 MB, 663s
full noise | B2 65 1000 0.065 ± 0.0078
1000/1000 shots, 12 failures, peak worker RAM 661 MB, 304s
no idle noise | bare 12 1000 0.012 ± 0.0034
1000/1000 shots, 35 failures, peak worker RAM 668 MB, 564s
no idle noise | original SD 35 1000 0.035 ± 0.0058
1000/1000 shots, 17 failures, peak worker RAM 668 MB, 575s
no idle noise | B1 17 1000 0.017 ± 0.0041
1000/1000 shots, 14 failures, peak worker RAM 668 MB, 602s
no idle noise | B2 14 1000 0.014 ± 0.0037
